### ЗАДАЧА: Реестр абонементов фитнес-клуба

Администратор фитнес-клуба получает строки с данными об абонементах.
Нужно собрать удобную модель, которая позволит:
- загрузить клиентов в единый реестр,
- посмотреть только активные абонементы,
- отфильтровать клиентов по тарифу,
- посчитать суммарное число оставшихся посещений,
- понять, как меняется реестр после активации и списания посещения.

В данных есть абонементы с разными статусами и остатком посещений,
поэтому важно корректно валидировать тариф, статус и изменение состояния объекта.


In [ ]:
# rows: sub_id|client_name|plan|visits_left|status
rows = [
    'SB-100|Alice|standard|8|active',
    'SB-101|Bob|premium|12|frozen',
    'SB-102|Charlie|family|0|expired',
    'SB-103|Diana|standard|5|active',
]


class Subscription:
    allowed_plans = {'standard', 'premium', 'family'}
    allowed_statuses = {'active', 'frozen', 'expired'}

    def __init__(self, sub_id, client_name, plan, visits_left, status):
        # TODO: сохранить sub_id, client_name, plan
        # TODO: visits_left хранить через self._visits_left
        # TODO: значение visits_left пропустить через property/setter
        # TODO: проверить plan и status, иначе raise ValueError
        self.sub_id = sub_id
        self.client_name = client_name
        if plan not in self.allowed_plans:
            raise ValueError("Недействительный план")
        self.plan = plan
        self._visits_left = None  
        self.visits_left = visits_left 
        if status not in self.allowed_statuses:
            raise ValueError("Недопустимый статус")
        self.status = status

    @property
    def visits_left(self):
        # TODO: вернуть текущее число посещений
        return self._visits_left

    @visits_left.setter
    def visits_left(self, value):
        # TODO: привести value к int
        # TODO: если value < 0 -> raise ValueError('Visits must be >= 0')
        # TODO: сохранить результат в self._visits_left
        value_int = int(value)
        if value_int < 0:
            raise ValueError("Количество посещений должно быть >= 0")
        self._visits_left = value_int

    def use_visit(self):
        # TODO: если статус не 'active' -> raise ValueError
        # TODO: если visits_left == 0 -> raise ValueError
        # TODO: уменьшить visits_left на 1
        # TODO: если после списания visits_left == 0, перевести статус в 'expired'
        if self.status != 'active':
            raise ValueError("Невозможно использовать visit: подписка не активна")
        if self.visits_left == 0:
            raise ValueError("Визитов больше не осталось")
        self.visits_left -= 1
        if self.visits_left == 0:
            self.status = 'expired'

    def freeze(self):
        # TODO: если статус 'expired' -> raise ValueError
        # TODO: перевести абонемент в 'frozen'
        if self.status == 'expired':
            raise ValueError("Невозможно заморозить подписку с истекшим сроком действия")
        self.status = 'frozen'


    def activate(self):
        # TODO: если visits_left == 0 -> raise ValueError
        # TODO: перевести абонемент в 'active'
        if self.visits_left == 0:
            raise ValueError("Не удается активировать подписку при 0 посещениях")
        self.status = 'active'

    @classmethod
    def from_row(cls, row):
        # TODO: split по '|'
        # TODO: ожидать 5 частей: sub_id, client_name, plan, visits_left, status
        # TODO: вернуть Subscription(...)
        parts = row.split('|')
        if len(parts) != 5:
            raise ValueError("Недопустимая строка")
        sub_id, client_name, plan, visits_left, status = parts
        return cls(sub_id, client_name, plan, visits_left, status)

    def __repr__(self):
        # TODO: вернуть строку вида Subscription(sub_id='...', client_name='...', status='...')
        return (f"Subscription(sub_id='{self.sub_id}', client_name='{self.client_name}', "
                f"plan='{self.plan}', visits_left={self.visits_left}, status='{self.status}')")


class SubscriptionRegistry:
    def __init__(self):
        self.items = []

    def add(self, subscription):
        # TODO: добавить subscription в self.items
        self.items.append(subscription)

    def load(self, rows):
        # TODO: для каждой строки создать Subscription.from_row(row)
        # TODO: добавить объект в реестр через add(...)
        for row in rows:
            sub = Subscription.from_row(row)
            self.add(sub)

    def active_subscriptions(self):
        # TODO: вернуть список абонементов со статусом 'active'
        return [sub for sub in self.items if sub.status == 'active']

    def by_plan(self, plan):
        # TODO: вернуть список абонементов нужного тарифа
        return [sub for sub in self.items if sub.plan == plan]

    def total_visits_left(self):
        # TODO: вернуть суммарное число оставшихся посещений
        return sum(sub.visits_left for sub in self.items)

    def status_summary(self):
        # TODO: собрать dict вида status -> count
        summary = {}
        for sub in self.items:
            summary[sub.status] = summary.get(sub.status, 0) + 1
        return summary

    def find(self, sub_id):
        # TODO: вернуть абонемент по sub_id или None
        for sub in self.items:
            if sub.sub_id == sub_id:
                return sub
        return None


registry = SubscriptionRegistry()

# TODO: загрузить rows в registry
registry.load(rows)

# TODO: вывести все абонементы
print("Все абонементы:")
for sub in registry.items:
    print(sub)


# TODO: вывести active_subscriptions()
print("\nАктивные абонементы:")
for sub in registry.active_subscriptions():
    print(sub)

# TODO: вывести by_plan('standard')
print("\nАбонементы по плану 'standard':")
for sub in registry.by_plan('standard'):
    print(sub)

# TODO: вывести total_visits_left()
print("\nОбщее число оставшихся посещений:", registry.total_visits_left())

# TODO: вывести status_summary()
print("\nСтатусный отчёт:", registry.status_summary())

# TODO: найти абонемент 'SB-101', активировать его и вывести status_summary()
sub_bob = registry.find('SB-101')
if sub_bob:
    sub_bob.activate()
    print("\nПосле активации SB-101:", sub_bob)
    print("Обновлённый отчёт:", registry.status_summary())

# TODO: найти абонемент 'SB-100', списать одно посещение и вывести объект
sub_alice = registry.find('SB-100')
if sub_alice:
    sub_alice.use_visit()
    print("\nПосле использования посещения SB-100:", sub_alice)